# 09 · RFM and customer profiling

**Fieldwise Cloud · Approved Phase 08 research · Great Britain**

Revenue Operations can use the profiles to understand gross billing
exposure, renewal cadence and variation across accounts. Customer
Success can review lifecycle and recorded product activity as possible
context for outreach. This fictional study does not establish
customer loyalty, intervention effects, or commercial segmentation.

**Evidence flow** · Canonical account → canonical contractual episodes
→ invoices, dated snapshots and episode-month usage → availability at the
UTC observation → evidence and eligibility checks → historical
measures and frozen financial references.

A predecessor keeps its invoice and usage lineage when a successor is
established. At an account observation, concurrent episodes can belong
to the same customer. Financial attribution follows invoice identity.
Usage aggregation has a separate ambiguity check.

Refer to the [approved Phase 08 decision record](../../docs/05_ml/phase08_segmentation_scope_and_validation.md)
and the [clustering and validation notebook](10_customer_clustering_and_validation.ipynb).

In [ ]:
from dataclasses import asdict
from datetime import UTC, datetime
from decimal import Decimal
from os import getenv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter

from nowa_customer_intelligence.models.clv.history import historical_value
from nowa_customer_intelligence.models.segmentation.diagnostics import numeric_quality
from nowa_customer_intelligence.models.segmentation.population import (
    development_populations,
    reference_evidence,
)
from nowa_customer_intelligence.models.segmentation.profiles import profile_accounts, window_start
from nowa_customer_intelligence.models.segmentation.research import DEFAULT_WORKSPACE, load_report
from nowa_customer_intelligence.models.segmentation.rfm import ReferenceScores
from nowa_customer_intelligence.models.segmentation.usage import UsageState

plt.rcParams.update(
    {
        "figure.figsize": (9, 4),
        "figure.dpi": 125,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.alpha": 0.2,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)
palette = {"monthly": "#236584", "annual": "#a45324", "mixed": "#68468a", "unobserved": "#606c70"}
labels = {
    "recency_days": "Invoice recency (days)",
    "frequency": "Invoices (count)",
    "monetary_gbp": "Gross issuance (GBP)",
    "tenure_days": "Tenure (days)",
}
columns = ["recency_days", "frequency", "monetary_gbp"]


def show(fig):
    fig.tight_layout()
    plt.show()
    plt.close(fig)


source = reference_evidence()
ledger = source.ledger
inventory = source.provenance["files"]
populations, audit = development_populations(source)
observed = populations[-1].observation

root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "pyproject.toml").is_file()
)
configured = getenv("NCI_SEGMENTATION_WORKSPACE")
workspace = Path(configured) if configured else root / DEFAULT_WORKSPACE
report = load_report(workspace) if configured or workspace.exists() else None
references = ReferenceScores.from_record(report["design"]["rfm_boundaries"]) if report else None


def account_frame(instant):
    rows = profile_accounts(ledger, instant)
    return pd.DataFrame.from_records(asdict(row) for row in rows).set_index(
        "customer_id", verify_integrity=True
    )


accounts = account_frame(observed)
rfm = accounts[columns].astype(float)

## Executive dashboard and source inventory

One account can hold several subscription episodes. A known reactivation
establishes a successor episode while retaining the predecessor and its
invoice attribution. Concurrent episodes are distinct contracts under the
same account. The historical ledger checks ownership and carries each
invoice once. Currency aggregation is exact Decimal GBP. Floating-point
conversions below are for charts and descriptive correlations.

Account → episode → issued invoice → evidence available at observation.
Lifecycle facts and snapshots are checked separately for their effective
and availability times. An invoice with failed payment can still be an
issue. Issued value measures gross billing exposure. Cash received,
recognised revenue, margin and economic lifetime value have separate
definitions. Snapshot GBP/month describes the observed contractual stock.

In [ ]:
cadence_order = [name for name in palette if name in accounts["cadence"].values]
gbp = sum(accounts["monetary_gbp"], Decimal("0"))
complete = accounts["calendar_history_complete"]
zero = accounts["invoice_feed_available"] & accounts["frequency"].eq(0)
dashboard = pd.DataFrame(
    {
        "Measure": [
            "UTC observation",
            "Visible accounts",
            "Established episodes",
            "Completed episodes",
            "Accounts with active episodes",
            "Reactivated episodes",
            "Complete calendar histories",
            "Observed zero issues",
            "Trailing issued GBP",
        ],
        "Value": [
            observed.isoformat(),
            len(accounts),
            accounts.established_episodes.sum(),
            accounts.completed_episodes.sum(),
            accounts.active_episodes.gt(0).sum(),
            accounts.reactivated_episodes.sum(),
            int(complete.sum()),
            int(zero.sum()),
            f"£{gbp:,.2f}",
        ],
    }
)
display(dashboard)
display(
    pd.DataFrame([{"Dataset": item["dataset"], "Records": item["records"]} for item in inventory])
)
display(
    accounts.groupby("cadence", observed=True)
    .agg(
        Accounts=("cadence", "size"),
        Active=("active_episodes", "sum"),
        Completed=("completed_episodes", "sum"),
        Reactivated=("reactivated_episodes", "sum"),
        Complete=("calendar_history_complete", "sum"),
        Trailing_GBP=("monetary_gbp", "sum"),
    )
    .reindex(cadence_order)
)

## Lifecycle and observed exposure

Completed episodes and currently active episodes can contribute to the
same customer profile. An account may also have no active episode and
recent invoices. Tenure is elapsed calendar time since the canonical
account creation. A complete calendar window requires the account and
source history to reach the lower boundary. Primary modelling also requires an intact verified five-export source and all three financial measures. This is a measure of observed
exposure within the fictional source. Production completeness would require independent evidence. Earlier landmarks can left-censor account histories; late
availability can change what is observable inside an otherwise long window.

In [ ]:
lifecycle = accounts.assign(
    footing=np.where(accounts.active_episodes.gt(0), "Known active", "No known active"),
    coverage=np.where(complete, "Complete", "Short history"),
)
order = ["Known active", "No known active"]
display(pd.crosstab(lifecycle["footing"], lifecycle["coverage"]).reindex(order, fill_value=0))
display(pd.crosstab(accounts["established_episodes"], accounts["reactivated_episodes"]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pd.crosstab(lifecycle.footing, lifecycle.coverage).reindex(order, fill_value=0).plot.bar(
    stacked=True,
    color=["#236584", "#d9a56e"],
    rot=0,
    ax=axes[0],
    title="Calendar coverage by contractual footing",
    ylabel="Accounts",
)
accounts.tenure_days.plot.hist(
    bins=24, color="#236584", ax=axes[1], title="Distribution of observed account tenure"
)
axes[1].set(xlabel="Tenure (days)", ylabel="Accounts")
show(fig)

rates = accounts.loc[accounts.active_episodes.gt(0), "contractual_gbp_per_month"]
display(rates.value_counts(dropna=False).rename_axis("GBP/month").to_frame("Accounts"))
fig, ax = plt.subplots()
counts = rates.astype(str).value_counts().sort_index()
ax.bar(counts.index, counts.values, color="#236584", width=0.55)
ax.set(
    title="Usable monthly contractual rates among known active accounts",
    xlabel="GBP/month",
    ylabel="Accounts",
)
show(fig)

## Approved RFM definitions

Let \(t\) be the UTC observation and \(s=t-12\) calendar months. For customer
\(i\), let \(I_i(t)\) be the distinct issues visible at \(t\), accounting for
both issue and evidence availability. An issue date is represented at UTC
midnight. The analytical window is open at \(s\) and closed at \(t\).

$$
R_i=\operatorname{date}(t)-\max_{j\in I_i(t)} d_j, \qquad
F_i=\sum_{j\in I_i(t)}\mathbf 1(s<d_j\le t), \qquad
M_i=\sum_{j\in I_i(t)} a_j\mathbf 1(s<d_j\le t).
$$

Recency uses the most recent visible issue even if it predates the window.
Recency is undefined without a visible issue. A present feed can support
zero observed issuance; a missing feed yields unknown frequency and
monetary value. Amount \(a_j\) is the original gross invoice amount in
GBP. Annual amounts remain whole. Exposure flags accompany the values.

The ordinal references put smaller recency and larger frequency and value
towards higher scores. For sorted development values, look up the observations
at ranks ⌈n/3⌉ and ⌈2n/3⌉. Equality stays in the lower raw band, duplicate
cuts and undersized bands are pooled, and exterior values use the outer
frozen band. With two effective levels use 1 and 3. With one use 2. Three
accounts with one invoice receive the same frequency score within a scoring
rule. The cadence-aware reference freezes a separate rule for each group
with at least 30 eligible development accounts.

In [ ]:
view = historical_value(ledger, observed)
start = window_start(observed)
example_id = (
    accounts.loc[complete & accounts.reactivated_episodes.gt(0) & accounts.cadence.eq("mixed")]
    .sort_values(["monetary_gbp", "tenure_days"])
    .index[0]
)
visible = [issue for issue in view.issues if issue.customer_id == example_id]
in_window = [
    issue
    for issue in visible
    if start < datetime.combine(issue.issued_on, datetime.min.time(), UTC)
]
episode_keys = {
    episode.subscription_id: f"Episode {index}"
    for index, episode in enumerate(
        sorted(
            (row for row in view.episodes if row.customer_id == example_id),
            key=lambda row: row.established_on,
        ),
        1,
    )
}
example = accounts.loc[example_id]
assert example.frequency == len(in_window)
assert example.monetary_gbp == sum((issue.issued_gbp for issue in in_window), Decimal("0"))
assert example.recency_days == (observed.date() - max(event.issued_on for event in visible)).days
display(
    pd.DataFrame(
        [
            {
                "Episode": episode_keys[issue.subscription_id],
                "Issue date": issue.issued_on,
                "Known on (UTC)": issue.available_at.date(),
                "Cadence": issue.cadence.value,
                "In window": issue in in_window,
                "Issued GBP": issue.issued_gbp,
            }
            for issue in visible
        ]
    )
)
display(
    Markdown(
        f"The worked customer has **{example.recency_days} days** since its last visible "
        f"issue, **{example.frequency} in-window invoices** and "
        f"**£{example.monetary_gbp:,.2f}** in trailing issuance."
    )
)

if references is not None:
    reference_profiles = {row.customer_id: row for row in populations[-1].eligible}
    worked = [
        ("Worked mixed account", reference_profiles[example_id]),
        (
            "Comparison monthly account",
            next(row for row in reference_profiles.values() if row.cadence == "monthly"),
        ),
    ]
    display(
        pd.DataFrame(
            [
                {
                    "Case": name,
                    "Observation": observed.date(),
                    "Window starts after": start.date(),
                    "Cadence": row.cadence,
                    "Recency days": row.recency_days,
                    "Invoices": row.frequency,
                    "Issued GBP": row.monetary_gbp,
                    "Traditional R, F, M": references.assign(row),
                    "Cadence-aware R, F, M": references.assign(row, cadence_aware=True)
                    or "Unassignable",
                }
                for name, row in worked
            ]
        )
    )

fig, ax = plt.subplots(figsize=(9.6, 4.6))
for label, colour in ((False, "#606c70"), (True, "#236584")):
    plotted = [issue for issue in visible if (issue in in_window) == label]
    ax.scatter(
        [issue.issued_on for issue in plotted],
        [float(issue.issued_gbp) for issue in plotted],
        color=colour,
        s=55,
        label="Inside window" if label else "Before window",
    )
ax.axvline(start.date(), color="#a45324", ls="--", label="Open lower boundary")
ax.axvline(observed.date(), color="#236584", ls=":", label="UTC observation date")
ax.set(
    title="Worked mixed-cadence account · amounts by original issue date",
    xlabel="UTC issue date",
    ylabel="Gross issue amount (GBP)",
)
ax.legend(ncol=2)
fig.autofmt_xdate()
show(fig)

## Distributions, quantiles and customer concentration

The three raw measures have different units and different shapes.
An empirical cumulative distribution gives the fraction of known accounts
at or below a value, keeping jumps at invoice atoms. Histograms give
marginal counts. The income-style concentration curve here refers strictly
to cumulative gross issuance. The decile table gives account-rank
contribution with fractional allocation when tied accounts straddle a
boundary. The resulting amounts describe value distribution, with no
assignments to accounts or tiers.

In [ ]:
display(rfm.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).round(2))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, key in zip(axes, columns, strict=True):
    known = rfm[key].dropna()
    bins = np.arange(known.min() - 0.5, known.max() + 1.5) if key == "frequency" else 24
    ax.hist(known, bins=bins, color="#236584")
    ax.set(xlabel=labels[key], ylabel="Accounts")
fig.suptitle("Marginal distributions of raw RFM measures")
show(fig)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, key in zip(axes, columns, strict=True):
    known = rfm[key].dropna().sort_values()
    ax.step(known, np.arange(1, len(known) + 1) / len(known), where="post", color="#236584")
    ax.set(xlabel=labels[key], ylabel="At or below value")
    ax.yaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle("Empirical cumulative distributions")
show(fig)

In [ ]:
sorted_gbp = sorted(accounts.monetary_gbp.dropna())
size = len(sorted_gbp)
deciles = []
for band in range(10):
    lower = Decimal(band * size) / 10
    upper = Decimal((band + 1) * size) / 10
    contribution = sum(
        (
            amount * max(Decimal(0), min(upper, rank + 1) - max(lower, rank))
            for rank, amount in enumerate(sorted_gbp)
        ),
        Decimal(0),
    )
    deciles.append(contribution)
assert sum(deciles) == gbp
display(
    pd.DataFrame(
        {
            "Rank decile": range(1, 11),
            "Issued GBP": deciles,
            "Share": [f"{value / gbp:.1%}" for value in deciles],
        }
    )
)
cumulative = [0, *np.cumsum([float(value / gbp) for value in sorted_gbp])]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
population_share = np.arange(size + 1) / size
axes[0].plot(population_share, cumulative, color="#236584", label="Observed")
axes[0].plot([0, 1], [0, 1], ls="--", color="#606c70", label="Equal issuance")
axes[0].set(
    title="Cumulative gross issuance",
    xlabel="Share of ranked accounts",
    ylabel="Share of issued GBP",
)
axes[0].xaxis.set_major_formatter(PercentFormatter(1))
axes[0].yaxis.set_major_formatter(PercentFormatter(1))
axes[0].legend()
axes[1].bar(range(1, 11), [float(value / gbp) for value in deciles], color="#236584")
axes[1].set(
    title="Gross issuance by rank decile",
    xlabel="Rank decile, low to high",
    ylabel="Share of issued GBP",
    xticks=range(1, 11),
)
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
show(fig)

## Cadence and contract structure

Monthly issues recur with many more opportunities to count than annual
issues. An annual payer can have long invoice recency while still active
and contractually current. Reactivation and a change of cadence can produce
mixed histories. Tenure and complete-window status are useful alongside
amounts. Ordered RFM scores applied across cadence groups could largely
rank renewal structure. Invoice timing alone cannot measure engagement.

The trailing window is an invoice window that can cross episode boundaries.
Inspect high counts against lifecycle evidence before assuming every record
has a single steady-state schedule. Issuance also differs from an active
snapshot rate. A catalogue with only one historical plan cannot support a
comparison among plan tiers.


In [ ]:
display(
    rfm.assign(cadence=accounts.cadence)
    .groupby("cadence")
    .agg(["median", "min", "max"])
    .round(1)
    .reindex(cadence_order)
)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, key in zip(axes, columns, strict=True):
    values = [rfm.loc[accounts.cadence.eq(name), key].dropna() for name in cadence_order]
    box = ax.boxplot(values, tick_labels=cadence_order, patch_artist=True)
    for patch, name in zip(box["boxes"], cadence_order, strict=True):
        patch.set_facecolor(palette[name])
        patch.set_alpha(0.5)
    ax.set(ylabel=labels[key], xlabel="Historical issue cadence")
fig.suptitle("Cadence comparison with median, IQR and tail points")
show(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for cadence in cadence_order:
    subset = accounts.loc[accounts.cadence.eq(cadence)]
    for ax, x in zip(axes, ("tenure_days", "recency_days"), strict=True):
        ax.scatter(
            subset[x],
            subset.monetary_gbp.astype(float),
            s=18,
            color=palette[cadence],
            alpha=0.35,
            label=cadence,
        )
        ax.set(xlabel=labels[x], ylabel="Gross issuance (GBP)")
axes[0].set_title("Exposure and value")
axes[1].set_title("Invoice recency and value")
axes[1].legend(title="Historical cadence")
show(fig)

fig, ax = plt.subplots()
values = accounts.assign(reactivated=accounts.reactivated_episodes.gt(0))
pivot = pd.crosstab(values.cadence, values.reactivated, normalize="index").reindex(cadence_order)
pivot.rename(columns={False: "No known reactivation", True: "Known reactivation"}).plot.bar(
    stacked=True,
    rot=0,
    color=["#d9a56e", "#236584"],
    ax=ax,
    title="Known reactivation within each cadence",
    xlabel="Historical cadence",
    ylabel="Share of cadence group",
)
ax.yaxis.set_major_formatter(PercentFormatter(1))
show(fig)

known_invoices = {issue.invoice_id for issue in view.issues}
plan_counts = (
    pd.Series(
        [
            charge.plan_id
            for charge in ledger.charges
            if charge.invoice.invoice_id in known_invoices
        ],
        name="Plan recorded on visible issues",
    )
    .value_counts()
    .rename("Issues")
)
display(plan_counts.to_frame())
display(
    accounts.loc[
        accounts.frequency.gt(12) | (accounts.cadence.eq("annual") & accounts.frequency.gt(1)),
        ["cadence", "frequency", "reactivated_episodes"],
    ]
    .value_counts()
    .rename("Accounts")
    .to_frame()
)

## Evidence quality and business interpretation

Financial coverage is verified for this fictional reference. Usage
observations are separately assessed in the three fixed months below.
Measures with high repeated values or zero IQR often reflect the
catalogue or generator structure. Spearman correlation captures monotone
association among available pairs, taking account of tied ranks. It
cannot establish incremental information or causal effects.

In [ ]:
quality = numeric_quality(accounts)
display(quality.round(3))
figures = quality[["Largest atom share"]].dropna().sort_values("Largest atom share")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(figures.index, figures.iloc[:, 0], color="#236584")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set(title="Largest repeated value per candidate field", xlabel="Share of known values")
show(fig)

measures = [
    name for name in quality.index if name != "cadence" and quality.loc[name, "Distinct"] > 1
]
correlations = accounts[measures].astype(float).corr(method="spearman")
fig, ax = plt.subplots(figsize=(9.5, 7))
image = ax.imshow(correlations, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set(
    xticks=range(len(measures)),
    yticks=range(len(measures)),
    yticklabels=measures,
    title="Pairwise Spearman correlations among variable measures",
)
ax.set_xticklabels(measures, rotation=55, ha="right")
ax.grid(False)
fig.colorbar(image, ax=ax, label="Spearman correlation")
show(fig)

cadence_report = rfm.join(accounts.cadence).groupby("cadence").median()
monthly = cadence_report.loc["monthly"]
annual = cadence_report.loc["annual"]
display(
    Markdown(
        f"**March findings.** The {int(complete.sum()):,} complete calendar windows "
        f"cover {complete.mean():.1%} of visible accounts. Median issue counts are "
        f"{monthly.frequency:g} monthly and {annual.frequency:g} annual, with "
        f"median recencies of {monthly.recency_days:g} and {annual.recency_days:g} days. "
        f"The top gross-issuance rank decile represents {deciles[-1] / gbp:.1%} of the total."
    )
)

## Modelling eligibility and the historical roster

Primary eligibility needs an exhaustive fictional export, a complete account
history window and all three usable RFM measures. Observed history before
eligibility is still useful for descriptive analysis. Visible excluded
accounts are each counted once by their first reason. The eventual roster
is used only to count accounts not yet visible. March 2023 is the fitting
observation. December 2023 supports candidate comparison and March 2024
measures later outcomes of fixed descriptive rules.

The same account may match several month ends. Neither the observed
growth in the eligible roster nor growth in incidence of mixed cadence
establishes financial-segment migration on its own.

In [ ]:
dispositions = (
    pd.DataFrame(
        [
            {
                "Observation": row["date"],
                "Visible": row["visible"],
                "Not yet visible": row["not_yet_visible"],
                "Complete calendar": row["complete_calendar"],
                "Eligible": row["eligible"],
                "Excluded": row["visible_excluded_unassignable"],
                **row["dispositions"],
            }
            for row in audit
        ]
    )
    .fillna(0)
    .set_index("Observation")
)
assert (dispositions.Visible == dispositions.Eligible + dispositions.Excluded).all()
display(dispositions.drop(columns=["eligible"], errors="ignore"))
cadence_support = (
    pd.DataFrame([row["cadence_eligible"] for row in audit], index=dispositions.index)
    .fillna(0)
    .astype(int)
)
display(cadence_support)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
dispositions[["Eligible", "Excluded", "Not yet visible"]].plot.bar(
    ax=axes[0],
    stacked=True,
    rot=0,
    color=["#236584", "#d9a56e", "#a2a9ab"],
    title="Eligibility reconciles to the source roster",
    ylabel="Accounts",
)
cadence_support.plot.bar(
    ax=axes[1],
    stacked=True,
    rot=0,
    color=palette,
    title="Cadence support among eligible accounts",
    ylabel="Accounts",
)
axes[1].legend(title="Historical cadence")
show(fig)

## Traditional and cadence-aware score boundaries

A neutral label such as **3–1–1** expresses an ordered recency, frequency
and monetary profile. It does not combine ordinals into a commercial
priority. Each band below is the actual frozen March 2023 boundary, with
units and direction from the dictionary. The interval below each cut uses
an inclusive upper limit. Repeated values at an invoice or catalogue atom
carry the same score.

Annual and monthly cut points describe different denominators. In
particular, one annual invoice could be typical annual frequency even
when it is lower than most monthly counts.

In [ ]:
score_labels = ["Recency", "Frequency", "Monetary"]
if report:
    boundary_record = report["design"]["rfm_boundaries"]
    reference_groups = {"Traditional": boundary_record["traditional"], **boundary_record["cadence"]}
    display(
        pd.DataFrame(
            [
                {
                    "Reference group": group,
                    "Dimension": name,
                    "Raw cut points": rule["thresholds"],
                    "Score levels from low raw to high": rule["levels"],
                    "Effective levels": len(rule["levels"]),
                }
                for group, rules in reference_groups.items()
                for name, rule in zip(score_labels, rules, strict=True)
            ]
        )
    )
    score_snapshot = report["rfm"][0]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for component, ax in enumerate(axes):
        frequencies = (
            pd.DataFrame(
                {
                    method.replace("_", " ").capitalize(): values["per_dimension"][component]
                    for method, values in score_snapshot["methods"].items()
                }
            )
            .reindex(["1", "2", "3"])
            .fillna(0)
        )
        frequencies.plot.bar(ax=ax, rot=0, color=["#68468a", "#236584"], legend=component == 2)
        ax.set(title=score_labels[component], xlabel="Frozen score", ylabel="Assigned accounts")
    fig.suptitle("Development reference scores · 338 primary accounts")
    show(fig)

    fitting_profiles = populations[0].frame()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    for name, ax in zip(("monthly", "annual"), axes, strict=True):
        counts = (
            fitting_profiles.loc[fitting_profiles.cadence.eq(name), "frequency"]
            .value_counts()
            .sort_index()
        )
        ax.bar(counts.index, counts.values, width=0.65, color=palette[name])
        for cut in boundary_record["cadence"][name][1]["thresholds"]:
            ax.axvline(cut + 0.5, color="#606c70", ls="--", label=f"Split above {cut}")
        ax.set(
            title=f"{name.capitalize()} development · {counts.sum()} accounts",
            xlabel="In-window invoices (count)",
            ylabel="Accounts",
            xticks=counts.index,
        )
        ax.legend()
    fig.suptitle("Tied invoice counts and the frozen cadence-specific frequency bands")
    show(fig)
else:
    display(Markdown("Run the Phase 08 research command to display the frozen score evidence."))

## Reference coverage, agreement and financial profiles

Compare exact three-component agreement among accounts assigned by both
methods. The diagonal of the matrix shows unchanged triples and
off-diagonal cells show different financial reference descriptions.
Unsupported mixed-cadence accounts remain visible in the coverage table.

The conventional analysis spans a common GBP distribution. The
cadence-aware analysis instead shows a relative position inside a
billing structure. Comparing the same neutral label across the two
methods requires looking at original invoice amounts and cadence mix.

In [ ]:
if report:
    reference_use = pd.DataFrame(
        [
            {
                "Date": item["date"],
                "Traditional": item["methods"]["traditional"]["accounts"],
                "Cadence-aware": item["methods"]["cadence_aware"]["accounts"],
                "Common": item["compared"],
                "Exact agreement accounts": item["agreement"],
                "Exact agreement": item["agreement"] / item["compared"],
                "Cadence unassignable": item["methods"]["cadence_aware"]["unassignable_by_cadence"],
            }
            for item in report["rfm"]
        ]
    ).set_index("Date")
    display(reference_use)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    reference_use[["Traditional", "Cadence-aware"]].plot.bar(
        ax=axes[0],
        rot=0,
        color=["#236584", "#68468a"],
        title="Frozen financial-reference support",
        ylabel="Assigned accounts",
    )
    axes[1].plot(reference_use.index, reference_use["Exact agreement"], "o-", color="#a45324")
    axes[1].set(
        title="Exact score agreement on common assignments",
        xlabel="Observation",
        ylabel="Share of common assignments",
        ylim=(0, 1),
    )
    axes[1].yaxis.set_major_formatter(PercentFormatter(1))
    show(fig)

    last_scores = report["rfm"][-1]
    grid = pd.DataFrame(last_scores["contingency"]).fillna(0).astype(int)
    fig, ax = plt.subplots(figsize=(8.6, 6))
    im = ax.imshow(grid, cmap="Purples", aspect="auto")
    ax.set(
        title=f"March 2024 reference agreement · {last_scores['compared']} common accounts",
        xlabel="Cadence-aware R–F–M",
        ylabel="Traditional R–F–M",
        xticks=range(len(grid.columns)),
        yticks=range(len(grid)),
        xticklabels=grid.columns,
        yticklabels=grid.index,
    )
    ax.tick_params(axis="x", labelrotation=45)
    for i, j in zip(*np.where(grid.to_numpy() > 0), strict=True):
        ax.text(
            j,
            i,
            str(grid.iloc[i, j]),
            va="center",
            ha="center",
            color="white" if grid.iloc[i, j] > grid.to_numpy().max() / 2 else "#192a33",
        )
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="Accounts")
    show(fig)

In [ ]:
if report:
    group_profiles = pd.DataFrame(
        [
            {
                "Reference": method,
                "Score": label,
                "Accounts": group["accounts"],
                "Median recency": group["statistics"]["recency_days"]["50%"],
                "Median invoices": group["statistics"]["frequency"]["50%"],
                "Median GBP": group["statistics"]["monetary_gbp"]["50%"],
                "Total GBP": Decimal(group["issued_gbp"]),
                **{name: group["cadence"].get(name, 0) for name in palette},
            }
            for method, evidence in last_scores["methods"].items()
            for label, group in evidence["groups"].items()
        ]
    )
    display(group_profiles.drop(columns="unobserved").sort_values(["Reference", "Score"]))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.7), sharey=True)
    for (method, data), ax in zip(group_profiles.groupby("Reference"), axes, strict=True):
        accounting = data.set_index("Score")
        mix = accounting[list(palette)].div(accounting.Accounts, axis=0)
        mix.loc[:, mix.sum().gt(0)].plot.bar(
            stacked=True, ax=ax, color=palette, title=method.replace("_", " ").capitalize()
        )
        ax.set(xlabel="Frozen R–F–M label", ylabel="Cadence share", ylim=(0, 1))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.tick_params(axis="x", labelrotation=45)
    fig.suptitle("Cadence composition within reference profiles · March 2024")
    show(fig)

    comparison = reference_use.iloc[-1]
    display(
        Markdown(
            f"At the final date **{int(comparison['Exact agreement accounts']):,} of "
            f"{int(comparison.Common):,}** comparable accounts have the same triple. All "
            f"**{int(comparison.Traditional - comparison['Cadence-aware']):,}** unmatched "
            "traditional accounts are from the mixed cadence. That group was below "
            "the development support requirement."
        )
    )

## Point-in-time account usage and reporting lag

For March month end, the fixed window contains January, February and
March. A completed month is not observable before its record's actual
availability timestamp. The configuration specifies a one-day minimum
reporting lag. No earlier month replaces the March slot.

An account period is usable only if one known active episode covers the
whole calendar month and its dated row supplies active users, sessions
and events. Multiple episodes within a month are ambiguous even when a
successor starts immediately after its predecessor. Distinct-user union
and additive event or session counts would need different evidence.
An absent active episode indicates structural absence; partial evidence
and reporting gaps retain their own states.

In [ ]:
final_population = populations[-1]
periods = pd.DataFrame(
    [
        {"Month": period.month.strftime("%b %Y"), "State": period.state.value}
        for record in final_population.usage.values()
        for period in record.periods
    ]
)
coverage = pd.crosstab(periods.Month, periods.State).reindex(
    [
        period.month.strftime("%b %Y")
        for period in next(iter(final_population.usage.values())).periods
    ]
)
coverage = coverage.reindex(columns=[state.value for state in UsageState], fill_value=0)
assert coverage.sum(axis=1).eq(len(accounts)).all()
assert coverage.to_numpy().sum() == 3 * len(accounts)
coverage_table = coverage.assign(Total=coverage.sum(axis=1))
coverage_table.loc["Total"] = coverage_table.sum()
display(coverage_table)
fig, ax = plt.subplots(figsize=(10, 4.5))
coverage.loc[:, coverage.sum().gt(0)].plot.bar(
    stacked=True,
    rot=0,
    ax=ax,
    color={
        "observed": "#236584",
        "no_known_active_episode": "#a2a9ab",
        "multiple_episodes": "#68468a",
        "partial_episode_or_record": "#a45324",
        "awaiting_reporting": "#d9a56e",
    },
    title="Outcome of 3,000 expected account-months",
    xlabel="Expected calendar month",
    ylabel="Accounts",
)
ax.legend(title="As-of state", bbox_to_anchor=(1, 1), loc="upper left")
show(fig)

practice = next(
    record
    for record in final_population.usage.values()
    if any(period.counts is not None and 0 in period.counts for period in record.periods)
    and record.periods[-1].counts is None
)
display(
    pd.DataFrame(
        [
            {
                "UTC month": period.month,
                "Evidence": period.state.value,
                "Active users": period.counts[0] if period.counts is not None else None,
                "Sessions": period.counts[1] if period.counts is not None else None,
                "Events": period.counts[2] if period.counts is not None else None,
            }
            for period in practice.periods
        ]
    )
)
display(pd.Series(practice.features(), name="Worked account summary").to_frame())

An observed zero is a verified count for its period. Latest counts use
the last usable period. The arithmetic mean divides by observed periods.
Mixed exposure can make comparisons of those means misleading. The
change between the first and last expected months is undefined if
either endpoint is missing. Keep the missing result visible rather
than imputing zero or taking the change over a different interval.

In [ ]:
usage_frame = final_population.frame()
usage_fields = ["mean_active_users", "mean_sessions", "mean_events"]
support = pd.crosstab(usage_frame.cadence, usage_frame.observed_usage_periods)
display(support.rename_axis(columns="Usable months"))
display(
    usage_frame[
        usage_fields + ["change_active_users", "change_sessions", "change_events"]
    ].describe()
)
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.1))
for name, ax in zip(usage_fields, axes, strict=True):
    subset = usage_frame.loc[usage_frame.observed_usage_periods.eq(2), name].dropna()
    ax.hist(subset, bins=18, color="#236584")
    ax.set(
        xlabel=name.replace("_", " ").capitalize() + " (count/month)",
        ylabel="Accounts",
        title=f"{len(subset)} eligible accounts with two periods",
    )
show(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
support.plot.bar(
    ax=axes[0],
    rot=0,
    stacked=True,
    color=["#d9a56e", "#236584"],
    title="Usable month count by eligible cadence",
    ylabel="Accounts",
)
axes[0].legend(title="Usable months")
sample = usage_frame.loc[usage_frame.observed_usage_periods.eq(2)]
for cadence, values in sample.groupby("cadence"):
    axes[1].scatter(
        values.mean_sessions,
        values.monetary_gbp.astype(float),
        s=14,
        alpha=0.43,
        color=palette[cadence],
        label=cadence,
    )
axes[1].set(
    title="Session counts and billed exposure in two comparable months",
    xlabel="Mean sessions (count/month)",
    ylabel="Trailing gross issued GBP",
)
axes[1].legend(title="Historical cadence")
show(fig)

coverage_by_date = pd.DataFrame(
    [
        {
            "Observation": row["date"],
            "Primary eligible": row["eligible"],
            "With any usable usage": row["eligible_with_any_usage"],
            "With all three usable periods": row["eligible_with_full_usage"],
        }
        for row in audit
    ]
)
display(coverage_by_date)

Partial usage exposure supports descriptive comparisons only. The fixed
window has no complete case across the inspected cut-offs, so FS-03
fails the entry gate. The absence of a defensible behavioural clustering
block does not turn the financial groups into an engagement measure.

The March 2024 audit reconciled all 3,000 expected account-months. The
earlier summary omitted 82 periods with no known active episode and
78 periods with partial coverage. No period fell into the missing or
unavailable category after applying the recorded statuses. The one-day
publication lag applies to the closing month within the fixed window.
Partial coverage and ambiguous episodes reduce usable exposure further.

Three latest complete and available reporting months are a deferred
investigation subject to separate Project Owner approval. The notebook
applies the accepted fixed calendar window.

## Feature dictionary and business reading

Source, units and availability matter before sorting accounts by value.
The modelling registry uses recency, invoice count and issued GBP. Tenure,
reactivation and cadence enter only the contractual comparisons.
Point-in-time data quality stays separate from customer characteristics.
Candidate usage fields remain documented so a later study can retest
availability without redefining these historical windows.

In [ ]:
if report:
    feature_table = pd.DataFrame(report["design"]["feature_dictionary"])
    with pd.option_context("display.max_colwidth", None):
        display(feature_table[["name", "calculation", "meaning", "source", "units", "dtype"]])
        display(
            feature_table[["name", "availability", "missing", "limitation", "spaces", "admission"]]
        )

## Conclusions and reproducibility

Revenue Operations could examine cadence-specific billing exposure
directly using the original amounts. Customer Success could inspect
episodes and reported activity as separate evidence. Neither use
warrants an automated decision on retention or engagement.

The Fieldwise catalogue and billing calendar strongly structure observed
frequency and amounts. RFM ranks are interpretable financial references.
Mixed cadence needs enough eligible development support for its own
reference. A complete behavioural study needs dated, complete reports
available at the decision timestamp, with validated cross-episode
attribution.

Reproduce in a fresh Python 3.12 notebook kernel using the offline accepted
seed-42 source, reconstructed in temporary storage. The analytical
copies read the verified Phase 08 report and apply frozen scoring rules.
They perform no clustering refit or reserve evaluation. The checked-in
sources keep outputs and execution counts cleared. Read the
[notebook execution notes](../README.md#phase-08-customer-segmentation).